In [1]:
import pandas as pd

In [2]:
from pyspark.sql import SparkSession

In [3]:
machinelearn=SparkSession.builder.appName('Practise').getOrCreate()

In [10]:
df_py=machinelearn.read.csv('test6.csv',header=True,inferSchema=True)

In [11]:
df_py.columns

['Employee_ID',
 'Name',
 'Age',
 'Gender',
 'Experience_Years',
 'Department',
 'Education',
 'Salary',
 'City',
 'Performance_Rating',
 'Remote_Work',
 'Projects_Handled',
 'Overtime_Hours',
 'Promotion',
 'Joined_Year']

In [12]:
df_py.show()

+-----------+-------+---+------+----------------+----------+---------+------+---------+------------------+-----------+----------------+--------------+---------+-----------+
|Employee_ID|   Name|Age|Gender|Experience_Years|Department|Education|Salary|     City|Performance_Rating|Remote_Work|Projects_Handled|Overtime_Hours|Promotion|Joined_Year|
+-----------+-------+---+------+----------------+----------+---------+------+---------+------------------+-----------+----------------+--------------+---------+-----------+
|       1001|  Ahmed| 25|  Male|               2|      Data|Bachelors|  5000|    Dubai|              Good|        Yes|               3|            12|       No|       2022|
|       1002|  Sarah| 30|Female|               5|        HR|  Masters|  8500|  Sharjah|         Excellent|         No|               6|            20|      Yes|       2019|
|       1003|   John| 28|  Male|               3|        IT|Bachelors|  7000|Abu Dhabi|           Average|        Yes|               4|

In [13]:
df_py.columns

['Employee_ID',
 'Name',
 'Age',
 'Gender',
 'Experience_Years',
 'Department',
 'Education',
 'Salary',
 'City',
 'Performance_Rating',
 'Remote_Work',
 'Projects_Handled',
 'Overtime_Hours',
 'Promotion',
 'Joined_Year']

In [21]:
from pyspark.ml.feature import StringIndexer
indexer=StringIndexer(inputCols=["Gender","Remote_work","Promotion","Education"],outputCols=["gender_indexed","remote_indexed","promotion_indexed","education_indexed"])

In [22]:
df_r=indexer.fit(df_py).transform(df_py)

In [23]:
from pyspark.ml.feature import VectorAssembler

In [31]:
feature_assembler=VectorAssembler(inputCols=["gender_indexed","remote_indexed","promotion_indexed","education_indexed","Age","Experience_Years"], outputCol="independent Feature")

In [32]:
output=feature_assembler.transform(df_r)

In [34]:
output.show()

+-----------+-------+---+------+----------------+----------+---------+------+---------+------------------+-----------+----------------+--------------+---------+-----------+--------------+--------------+-----------------+-----------------+--------------------+
|Employee_ID|   Name|Age|Gender|Experience_Years|Department|Education|Salary|     City|Performance_Rating|Remote_Work|Projects_Handled|Overtime_Hours|Promotion|Joined_Year|gender_indexed|remote_indexed|promotion_indexed|education_indexed| independent Feature|
+-----------+-------+---+------+----------------+----------+---------+------+---------+------------------+-----------+----------------+--------------+---------+-----------+--------------+--------------+-----------------+-----------------+--------------------+
|       1001|  Ahmed| 25|  Male|               2|      Data|Bachelors|  5000|    Dubai|              Good|        Yes|               3|            12|       No|       2022|           0.0|           1.0|              1.0|

In [33]:
output.select('independent feature').show()

+--------------------+
| independent feature|
+--------------------+
|[0.0,1.0,1.0,1.0,...|
|[1.0,0.0,0.0,0.0,...|
|[0.0,1.0,1.0,1.0,...|
|[1.0,0.0,0.0,2.0,...|
|[0.0,1.0,1.0,1.0,...|
|[1.0,0.0,0.0,0.0,...|
|[0.0,0.0,0.0,2.0,...|
|[1.0,1.0,1.0,1.0,...|
|(6,[4,5],[29.0,5.0])|
|[1.0,1.0,0.0,2.0,...|
|[0.0,1.0,1.0,3.0,...|
|[1.0,0.0,0.0,0.0,...|
|[0.0,0.0,0.0,2.0,...|
|[0.0,1.0,1.0,1.0,...|
|[1.0,1.0,1.0,0.0,...|
|[0.0,0.0,0.0,2.0,...|
|[1.0,1.0,1.0,1.0,...|
|(6,[4,5],[34.0,8.0])|
|[1.0,1.0,0.0,0.0,...|
|[0.0,0.0,0.0,4.0,...|
+--------------------+
only showing top 20 rows


In [36]:
finalized_data=output.select('independent feature','Salary')

In [37]:
from pyspark.ml.regression import LinearRegression

In [38]:
train_data,test_data=finalized_data.randomSplit([0.75,0.25])

In [39]:
regressor=LinearRegression(featuresCol='independent feature',labelCol='Salary')

In [40]:
regressor=regressor.fit(train_data)

In [41]:
pred_results=regressor.evaluate(test_data)

In [42]:
pred_results.predictions.show()

+--------------------+------+-----------------+
| independent feature|Salary|       prediction|
+--------------------+------+-----------------+
|[0.0,1.0,1.0,0.0,...|  8900|9219.360771204238|
|[0.0,1.0,1.0,1.0,...|  5000|5511.752995856256|
|[0.0,1.0,1.0,1.0,...|  6500|6881.481899443001|
+--------------------+------+-----------------+

